# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, describing its structure and content.

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# `to_json()` produces a nested dict structure of metadata
metadata = dataset.metadata.to_json()

# Display dataset overview
# metadata must be treated as a single object: access top-level fields directly without iterating or subscripting
print(f"{metadata.get('name', 'Dataset')}: {metadata.get('description', '')}")

# Print dataset citation, publication, keywords
print("\nCitation:", metadata.get('citeAs', ''))
print("Published:", metadata.get('datePublished', ''))
print("Keywords:", metadata.get('keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# The Croissant schema defines record sets with unique '@id'.
# List all record sets and fields (@id) available in the dataset.

record_sets = dataset.metadata.record_sets
print("Record Sets and their fields:")

all_record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    all_record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', 'Unknown')}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - Column @id: {col.id} | name: {col.name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into pandas DataFrames
# Use the @id values from previous overview

dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"--- DataFrame for RecordSet @id {record_set_id} ---")
    print("Columns:", df.columns.tolist())
    print("Head:")
    print(df.head())
    print("\n")

# Choose one record set for further analysis
if len(all_record_set_ids) > 0:
    main_record_set_id = all_record_set_ids[0]
else:
    main_record_set_id = None

# If available, show its shape
if main_record_set_id is not None:
    print(f"Shape of selected RecordSet '{main_record_set_id}' DataFrame:", dataframes[main_record_set_id].shape)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Let's select a numeric field for analysis from the chosen record set.
# Use field @id from section 2 (example: Age, disease interval, etc.)
df = dataframes[main_record_set_id]

# Find a numeric column by looking at the schema fields
main_rs = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break

numeric_field_id = None
numeric_field_name = None
for field in getattr(main_rs, 'fields', []):
    if getattr(field, 'data_type', None) in ['Integer', 'Float', 'Number']:
        numeric_field_id = field.id
        numeric_field_name = field.name
        break

if numeric_field_name is None:
    # fallback: try to locate a likely numeric field
    possible_numeric_cols = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower()]
    if possible_numeric_cols:
        numeric_field_name = possible_numeric_cols[0]
        numeric_field_id = numeric_field_name

print(f"Selected numeric field: {numeric_field_name} (recorded as @id: {numeric_field_id})")

# Filter on numeric field > threshold
threshold = 10
if numeric_field_name is not None:
    filtered_df = df[df[numeric_field_name] > threshold]
    print(f"Filtered records with {numeric_field_name} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_name}_normalized"] = (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) / filtered_df[numeric_field_name].std()
    print(f"Normalized {numeric_field_name} for filtered records:")
    print(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

    # Find a group field for grouping (e.g., anatomical location, sex, MSI-H status)
    group_field_name = None
    for field in getattr(main_rs, 'fields', []):
        if getattr(field, 'data_type', None) == 'Text' and (
            'anatomical' in field.name.lower() or 'sex' in field.name.lower() or 'msi' in field.name.lower()):
            group_field_name = field.name
            break
    if group_field_name is None:
        # fallback
        possible_group_cols = [col for col in df.columns if 'location' in col.lower() or 'sex' in col.lower() or 'msi' in col.lower()]
        if possible_group_cols:
            group_field_name = possible_group_cols[0]

    print(f"Selected group field: {group_field_name}")

    # Group by group_field if present
    if group_field_name is not None and group_field_name in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_name)[numeric_field_name].mean().reset_index()
        print(f"Grouped data by {group_field_name} (mean of {numeric_field_name}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
# Simple visualization: histogram of numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_name is not None and numeric_field_name in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(data=df, x=numeric_field_name, bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_name}")
    plt.xlabel(numeric_field_name)
    plt.ylabel("Frequency")
    plt.show()

# Bar plot: mean numeric field per group
if group_field_name is not None and group_field_name in df.columns:
    plt.figure(figsize=(8,4))
    group_means = df.groupby(group_field_name)[numeric_field_name].mean().sort_values()
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field_name} by {group_field_name}")
    plt.xlabel(group_field_name)
    plt.ylabel(f"Mean {numeric_field_name}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides detailed clinical and molecular data for 77 second primary colorectal cancer survivors.
- We've accessed metadata, reviewed structural entities using their `@id`, loaded record sets into DataFrames, and visualized numeric fields.
- Based on the selected numeric and grouping fields, we observed the distribution and group means, e.g., age or anatomical location.
- This notebook can be extended for deeper statistical analysis, predictive modeling, or further FAIR data processing using Croissant metadata.
